# F1 Analytics — Notebook 02: Linear Regression
**Race:** 2024 Japanese Grand Prix · Suzuka

---

## What this notebook covers
We continue the CRISP-DM process — moving from Data Understanding into **Modeling**.

From your course (Modeling II): *Linear Regression predicts a continuous value by finding the weighted combination of input features that best fits the data.*

---

## Business Understanding
**Question:** Can we predict a driver's lap time given how old their tires are and how far into the race they are?

**Why this matters:**  
F1 strategy is built on lap time prediction. Teams need to know: if we stay out 5 more laps on these tires, how much slower will we get? That decision — pit now or stay out — often decides the race. A linear regression model is the simplest tool to quantify tire degradation.

**From your EDA (notebook 01):**
- VER ran Medium (28 laps) + Hard (18 laps)
- PIA ran Medium (8 laps) + Hard (38 laps) — very different strategies
- VER median lap: 96.393s · PIA median lap: 97.021s
- VER was 0.628s faster per lap on median

The different strategies mean we have good data to see how each compound degrades.

---
## Step 1 — Imports

**What's new here compared to notebook 01?**  
We now import from `sklearn` — scikit-learn, the standard Python machine learning library. Three specific tools:

| Import | What it does |
|---|---|
| `LinearRegression` | The model itself — finds the best-fit line through the data |
| `train_test_split` | Splits data into training and testing sets (explained in Step 3) |
| `mean_squared_error`, `r2_score` | Metrics to measure how good our model is (explained in Step 6) |

**Why sklearn and not writing the math ourselves?**  
You could implement linear regression from scratch — it's just matrix algebra. But sklearn is the industry standard, it's what every data scientist uses at work, and it handles edge cases correctly. Using it is the right decision.

In [ ]:
%matplotlib inline
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("All libraries imported.")

---
## Step 2 — Reload the Data

**Why reload? Didn't we already load this in notebook 01?**  
Each notebook is independent — variables from notebook 01 don't carry over. When you open notebook 02 fresh, Python's memory is empty. We always reload from the cache (which is instant since the data is already downloaded).

This is actually good practice: each notebook should be self-contained and runnable on its own.

In [ ]:
fastf1.Cache.enable_cache('cache')

session = fastf1.get_session(2024, 'Japanese Grand Prix', 'R')
session.load()

# Reproduce the cleaning steps from notebook 01
laps = session.laps
clean_laps = laps.pick_accurate()

ver_laps = clean_laps.pick_drivers('VER').copy()
pia_laps = clean_laps.pick_drivers('PIA').copy()

ver_laps['LapTimeSec'] = ver_laps['LapTime'].dt.total_seconds()
pia_laps['LapTimeSec'] = pia_laps['LapTime'].dt.total_seconds()

print(f"Loaded: {len(ver_laps)} VER laps, {len(pia_laps)} PIA laps")

---
## Step 3 — Feature Engineering

**What is feature engineering?**  
From your course: a model needs numbers as input (features = X) and a number to predict (target = y). Feature engineering is the process of deciding which columns to use and transforming them into the right format.

**Our target (y) — what we want to predict:**  
`LapTimeSec` — the lap time in seconds.

**Our features (X) — what we use to predict it:**

| Feature | Column | Why it should predict lap time |
|---|---|---|
| Tire age | `TyreLife` | Older tires have less grip → slower laps. Direct physical relationship. |
| Race progress | `LapNumber` | Early laps: car is heavy with fuel → slower. As fuel burns off, car gets lighter → faster. |
| Tire compound | `Compound` | Medium tires are inherently faster than Hard tires at fresh age but degrade faster. |

**Why not use sector times as features?**  
Sector times add up to the lap time — using them to predict lap time would be circular (predicting something from its own components). That's called data leakage and it artificially inflates your model's performance.

**Why not use driver name as a feature?**  
We'll build separate models per driver instead. That way we can compare: does Verstappen's tire degradation slope differ from Piastri's?

**What is encoding and why do we need it?**  
`Compound` is a text value: `'MEDIUM'` or `'HARD'`. Machine learning models only understand numbers — they do math on inputs. We convert: `MEDIUM = 0`, `HARD = 1`. This is called label encoding.

In [ ]:
def prepare_features(driver_laps):
    """
    Takes a driver's lap DataFrame and returns X (features) and y (target).
    We put this in a function so we can reuse it for both VER and PIA.
    """
    df = driver_laps.copy()

    # Encode compound: MEDIUM=0, HARD=1
    # map() replaces each value using the dictionary
    df['Compound_encoded'] = df['Compound'].map({'MEDIUM': 0, 'HARD': 1})

    # Drop rows where any of our features or target is missing
    # subset= means: only drop if these specific columns have missing values
    df = df.dropna(subset=['LapTimeSec', 'TyreLife', 'LapNumber', 'Compound_encoded'])

    # X = feature matrix (what we feed into the model)
    # y = target vector (what the model tries to predict)
    X = df[['TyreLife', 'LapNumber', 'Compound_encoded']]
    y = df['LapTimeSec']

    return X, y, df

X_ver, y_ver, ver_df = prepare_features(ver_laps)
X_pia, y_pia, pia_df = prepare_features(pia_laps)

print("Verstappen features (first 5 rows):")
print(X_ver.head())
print()
print(f"Feature matrix shape: {X_ver.shape} → {X_ver.shape[0]} laps, {X_ver.shape[1]} features")

---
## Step 4 — Train/Test Split

**What is a train/test split and why do we need it?**  
From your course: *Overfitting = model memorizes training data but fails on new data.*

If we train the model on all our data and then measure its accuracy on the same data, of course it looks accurate — it already saw those answers. That's like giving a student the exam and then testing them on the same exam. Tells you nothing about whether they actually learned.

The split works like this:
```
All laps (46)
├── Training set (80% = ~37 laps) → model learns from these
└── Test set    (20% = ~9 laps)  → model is evaluated on these (never seen during training)
```

**Why 80/20 and not 50/50 or 90/10?**  
80/20 is the standard default. With only 46 laps, we need as many training examples as possible (so 80% train), but we also need enough test examples to get a meaningful accuracy score (so 20% test = ~9 laps). With smaller datasets, 80/20 is a reasonable balance.

**What does `random_state=42` mean?**  
The split is random — different random splits would give slightly different results. `random_state=42` fixes the random seed so you get the same split every time you run the notebook. This makes results reproducible. The number 42 is arbitrary — convention in the data science community (a joke reference to *The Hitchhiker's Guide to the Galaxy*).

In [ ]:
# Split VER data
# test_size=0.2 → 20% goes to test set, 80% to training set
X_ver_train, X_ver_test, y_ver_train, y_ver_test = train_test_split(
    X_ver, y_ver, test_size=0.2, random_state=42
)

# Split PIA data
X_pia_train, X_pia_test, y_pia_train, y_pia_test = train_test_split(
    X_pia, y_pia, test_size=0.2, random_state=42
)

print("Verstappen split:")
print(f"  Training laps: {len(X_ver_train)}")
print(f"  Test laps:     {len(X_ver_test)}")
print()
print("Piastri split:")
print(f"  Training laps: {len(X_pia_train)}")
print(f"  Test laps:     {len(X_pia_test)}")

---
## Step 5 — Do We Need to Standardize?

**From your course:** *Data Transformation — standardization scales features to mean=0, std=1. Required when algorithms depend on distance or assume normal distribution.*

**Do we need it here?**  
This is an important decision. The short answer: **no, not for linear regression**, and here's why:

| Situation | Need standardization? | Why |
|---|---|---|
| Linear Regression | **No** | It finds the best line regardless of scale. Coefficients are still correct. |
| K-Means (notebook 03) | **Yes** | Distance-based — large-scale features dominate |
| Neural Networks | **Yes** | Training is unstable without it |
| SVM | **Yes** | Kernel functions depend on distances |

There is one reason you *might* standardize for linear regression: if you want to compare coefficients to see which feature is *most important*. Without standardization, a coefficient of 0.5 on `TyreLife` (range 1–38) vs 0.5 on `LapNumber` (range 1–53) don't mean the same thing.

**Decision here:** We skip standardization because we want the coefficients in real, interpretable units — *"each extra lap on a tire adds X seconds"* is meaningful. If we standardized, the coefficients would be in abstract units with no real-world meaning.

This is the kind of decision you should be able to explain in an interview.

---
## Step 6 — Fit the Model

**What does `.fit()` actually do?**  
From your course: *Linear regression uses Ordinary Least Squares (OLS) — it finds the line that minimizes the sum of squared errors.*

In plain English: the model tries every possible set of weights for `TyreLife`, `LapNumber`, and `Compound_encoded`, and finds the combination that makes the predictions as close to the real lap times as possible. Mathematically it does this in one step (not trial-and-error).

The result is a formula like:
```
LapTimeSec = intercept + (w1 × TyreLife) + (w2 × LapNumber) + (w3 × Compound_encoded)
```

Where `w1`, `w2`, `w3` are the weights (coefficients) the model learned.

**What does `.predict()` do?**  
It takes new feature values (the test set the model never saw) and applies the formula above to generate predicted lap times.

In [ ]:
# Create model objects
model_ver = LinearRegression()
model_pia = LinearRegression()

# .fit() = train the model on the training data
# It reads X_train (features) and y_train (real answers) and finds the best weights
model_ver.fit(X_ver_train, y_ver_train)
model_pia.fit(X_pia_train, y_pia_train)

# .predict() = apply the learned formula to the test set
# These are the model's guesses for laps it never saw during training
y_ver_pred = model_ver.predict(X_ver_test)
y_pia_pred = model_pia.predict(X_pia_test)

print("Models trained. Predictions generated on test set.")
print()
print("Sample — VER test set (real vs predicted):")
comparison = pd.DataFrame({
    'Real LapTime': y_ver_test.values,
    'Predicted':    y_ver_pred.round(3),
    'Error':        (y_ver_pred - y_ver_test.values).round(3)
})
print(comparison.to_string(index=False))

---
## Step 7 — Evaluate the Model

**How do we know if the model is good?**  
Two standard metrics from your course:

**MSE — Mean Squared Error**
```
MSE = average of (real - predicted)²
```
- Measures average prediction error, but squared (so big errors are penalized more)
- Lower = better
- The unit is seconds² (because we squared it) — hard to interpret directly
- We also calculate RMSE (square root of MSE) to get back to seconds — easier to read

**R² — R-squared (Coefficient of Determination)**
```
R² = 1 means the model explains 100% of the variation in lap times
R² = 0 means the model is no better than just predicting the average every time
R² < 0 means the model is worse than just predicting the average
```
- Range: typically 0 to 1 (can go negative for very bad models)
- Higher = better
- Easier to interpret than MSE — 0.8 means "our 3 features explain 80% of why lap times vary"

**What R² value should we expect here?**  
We only have 3 features. There are many other things that affect lap time — track temperature, wind, traffic, driver errors, fuel load changes. A R² of 0.6–0.8 would be good for this simple model. Perfect (1.0) would mean something is wrong (data leakage).

In [ ]:
def evaluate_model(y_real, y_pred, driver_name):
    mse  = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)           # back to seconds (square root undoes the squaring)
    r2   = r2_score(y_real, y_pred)

    print(f"=== {driver_name} ===")
    print(f"  MSE:  {mse:.4f} s²")
    print(f"  RMSE: {rmse:.4f} s   ← average prediction error in seconds")
    print(f"  R²:   {r2:.4f}      ← model explains {r2*100:.1f}% of lap time variation")
    print()
    return rmse, r2

rmse_ver, r2_ver = evaluate_model(y_ver_test, y_ver_pred, 'Verstappen')
rmse_pia, r2_pia = evaluate_model(y_pia_test, y_pia_pred, 'Piastri')

---
## Step 8 — Interpret the Coefficients

**This is the most valuable part of linear regression.**  

The model learned weights (coefficients) for each feature. Because we didn't standardize, these are in real units — seconds per lap. You can read them like this:

- `TyreLife` coefficient = how many seconds slower per additional lap on the tire
- `LapNumber` coefficient = how many seconds faster/slower per lap further into the race
- `Compound_encoded` coefficient = how many seconds slower Hard tires are vs Medium at the same tire age
- `intercept` = the predicted lap time when all features are zero (not directly meaningful physically, but needed by the formula)

**Why is this useful for an interview answer?**  
"My model found that for Verstappen, each additional lap on a tire costs X seconds of pace. For Piastri it was Y seconds — meaning Verstappen managed tire degradation better/worse." That's a real insight from the model, not just a number.

In [ ]:
def print_coefficients(model, driver_name):
    feature_names = ['TyreLife', 'LapNumber', 'Compound (Hard vs Medium)']
    print(f"=== {driver_name} — Model Formula ===")
    print(f"  Intercept (baseline): {model.intercept_:.3f}s")
    print()
    for name, coef in zip(feature_names, model.coef_):
        direction = 'slower' if coef > 0 else 'faster'
        print(f"  {name:<30} {coef:+.4f}s per unit  → {abs(coef):.4f}s {direction} per unit increase")
    print()

print_coefficients(model_ver, 'Verstappen')
print_coefficients(model_pia, 'Piastri')

print("Interpretation guide:")
print("  TyreLife: positive = gets slower as tires age (expected)")
print("  LapNumber: negative = gets faster as fuel burns off (expected)")
print("  Compound: positive = Hard tires are slower than Medium at same age (expected)")

---
## Step 9 — Chart 1: Actual vs Predicted Lap Times

**What does this chart show?**  
Every dot is one lap from the test set. The x-axis = real lap time, y-axis = what the model predicted. 

**What is the diagonal line?**  
The perfect prediction line — if the model was 100% accurate, every dot would sit exactly on it. Dots above the line = model over-predicted (guessed slower than reality). Dots below = model under-predicted (guessed faster).

**Why plot this instead of just looking at the numbers?**  
The numbers (MSE, R²) tell you the average error. The chart tells you *where* the model is wrong — are errors clustered at fast laps? Slow laps? Randomly spread? A chart catches patterns that a single number hides.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot for each driver
for ax, y_real, y_pred, name, color in zip(
    axes,
    [y_ver_test, y_pia_test],
    [y_ver_pred, y_pia_pred],
    ['Verstappen', 'Piastri'],
    ['#0600EF', '#FF8000']
):
    # Scatter plot: each dot = one lap
    ax.scatter(y_real, y_pred, color=color, alpha=0.7, s=60, label='Predicted laps')

    # Perfect prediction line
    # We need the min and max across both real and predicted to draw it end-to-end
    line_min = min(y_real.min(), y_pred.min())
    line_max = max(y_real.max(), y_pred.max())
    ax.plot([line_min, line_max], [line_min, line_max],
            color='red', linestyle='--', linewidth=1.5, label='Perfect prediction')

    ax.set_xlabel('Real Lap Time (s)', fontsize=11)
    ax.set_ylabel('Predicted Lap Time (s)', fontsize=11)
    ax.set_title(f'{name} — Actual vs Predicted', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Linear Regression: How Well Does the Model Predict Lap Times?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/05_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10 — Chart 2: Tire Degradation Curves

**What is tire degradation?**  
As tires wear, they lose grip and lap times get slower. The rate at which this happens (the slope) differs by compound and driver.

**Why plot this separately from the regression output?**  
The regression model tells us *what the relationship is*. This chart shows it visually — you can see the degradation slope as a line through the real data points.

**Why split by compound?**  
Medium and Hard tires degrade at different rates. Mixing them in one chart would blur the picture. From EDA: VER ran mostly Medium, PIA mostly Hard — so this chart also shows which compound each driver relied on.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

compound_colors = {'MEDIUM': '#FFD700', 'HARD': '#AAAAAA'}  # F1 official compound colours

for ax, driver_df, name in zip(axes, [ver_df, pia_df], ['Verstappen', 'Piastri']):
    for compound in driver_df['Compound'].unique():
        subset = driver_df[driver_df['Compound'] == compound]

        # Scatter: real lap times coloured by compound
        ax.scatter(subset['TyreLife'], subset['LapTimeSec'],
                   color=compound_colors.get(compound, 'grey'),
                   label=compound, alpha=0.8, s=40)

        # Trend line: fit a simple 1D regression just for this compound
        # np.polyfit(x, y, 1) fits a degree-1 polynomial (a straight line)
        # It returns [slope, intercept]
        if len(subset) > 2:  # need at least 3 points to draw a meaningful line
            z = np.polyfit(subset['TyreLife'], subset['LapTimeSec'], 1)
            p = np.poly1d(z)  # turns [slope, intercept] into a callable function
            x_line = np.linspace(subset['TyreLife'].min(), subset['TyreLife'].max(), 50)
            ax.plot(x_line, p(x_line),
                    color=compound_colors.get(compound, 'grey'),
                    linewidth=2, linestyle='--')

    ax.set_xlabel('Tire Age (laps)', fontsize=11)
    ax.set_ylabel('Lap Time (seconds)', fontsize=11)
    ax.set_title(f'{name} — Tire Degradation by Compound', fontsize=13, fontweight='bold')
    ax.legend(title='Compound', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('2024 Japanese GP — Tire Degradation Curves',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/06_tire_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 11 — Chart 3: Residuals Plot

**What is a residual?**  
From your course: *Residual = real value − predicted value.* It's the error for each individual lap.

**Why plot residuals? Didn't we already evaluate with MSE and R²?**  
MSE gives you one average number. Residuals show you *the pattern* of errors. A good model should have residuals that are:
- Randomly scattered around zero (no pattern)
- Roughly the same spread across all predictions

If you see a pattern — like residuals that consistently go up as predicted time increases — that means the model is systematically wrong in a way that a more complex model could fix.

**What is the dashed line at y=0?**  
Zero error — where dots would sit if the model was perfect. Dots above = model predicted too slow. Dots below = predicted too fast.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_real, y_pred, name, color in zip(
    axes,
    [y_ver_test, y_pia_test],
    [y_ver_pred, y_pia_pred],
    ['Verstappen', 'Piastri'],
    ['#0600EF', '#FF8000']
):
    # Residual = real - predicted
    residuals = y_real.values - y_pred

    # x-axis: what the model predicted
    # y-axis: how wrong it was
    ax.scatter(y_pred, residuals, color=color, alpha=0.7, s=60)

    # Zero line — perfect prediction sits here
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)

    ax.set_xlabel('Predicted Lap Time (s)', fontsize=11)
    ax.set_ylabel('Residual (Real − Predicted) (s)', fontsize=11)
    ax.set_title(f'{name} — Residuals', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Residual Plot — Are Errors Random or Patterned?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/07_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print("If dots are randomly scattered around the dashed line → model errors are random → good.")
print("If dots form a curve or pattern → model is missing something → more complex model needed.")

---
## Step 12 — Summary

**What did linear regression tell us that EDA alone couldn't?**  
EDA showed us *that* Verstappen was faster. Linear regression tells us *why* in quantified terms:
- Exactly how much each lap on a tire costs in seconds
- Whether the fuel load effect (LapNumber) was significant
- How much slower Hard tires are vs Medium at the same age

These are the kind of insights an F1 strategy team actually uses.

In [ ]:
print("=" * 60)
print("LINEAR REGRESSION SUMMARY — 2024 Japanese GP")
print("=" * 60)
print()
print("Model performance (on test set — laps model never saw):")
print(f"  Verstappen → RMSE: {rmse_ver:.3f}s | R²: {r2_ver:.3f}")
print(f"  Piastri    → RMSE: {rmse_pia:.3f}s | R²: {r2_pia:.3f}")
print()
print("Tire degradation rates (seconds lost per lap on tire):")
print(f"  Verstappen: {model_ver.coef_[0]:+.4f}s per lap")
print(f"  Piastri:    {model_pia.coef_[0]:+.4f}s per lap")
print()
print("Fuel load effect (seconds gained per lap into the race):")
print(f"  Verstappen: {model_ver.coef_[1]:+.4f}s per lap")
print(f"  Piastri:    {model_pia.coef_[1]:+.4f}s per lap")
print()
print("Hard vs Medium penalty (seconds slower on Hard at same tire age):")
print(f"  Verstappen: {model_ver.coef_[2]:+.4f}s")
print(f"  Piastri:    {model_pia.coef_[2]:+.4f}s")
print()
print("Next: 03_clustering.ipynb — group all drivers by driving style using K-Means.")

---
## What We Used From Your Course

| Course concept | Where we used it | Why it mattered |
|---|---|---|
| **Linear Regression** | Steps 6–8 | Core model — predicts lap time from 3 features |
| **OLS / MSE** | Step 7 | How the model learns + how we measure error |
| **Overfitting / train-test split** | Step 4 | Evaluate on data the model never saw |
| **Feature engineering** | Step 3 | Chose meaningful inputs, encoded categorical variable |
| **Data Transformation decision** | Step 5 | Chose NOT to standardize — and explained why |
| **Residuals** | Step 11 | Check whether errors are random or patterned |

---
**Next notebook:** `03_clustering.ipynb` — K-Means clustering to group all 20 drivers by driving style.